
%md
###### 23_nlp_vector_search

###### Purpose
Create Vector Search endpoint and Vector Search Index for Support Ticket data. This will be used by Semantic Search and RAG.

###### Technologies Used

- Databricks

- Delta Lake

- Unity Catalog

- Databricks Vector Search

- Databricks Embedding Foundation Model (databricks-gte-large-en)

###### Input

- Create Vector Search Endpoint

- Delta table with precomputed embeddings

######  Output

- Retrieve top-k semantically similar support tickets.

######  Architecture

```text

Support Ticket Embeddings Delta Table
        ↓
Enable Change Data Feed
        ↓
Vector Search Endpoint
        ↓
Delta Sync Index
        ↓
User Question
        ↓
Same Embedding Model
        ↓
Question Embedding
        ↓
Similarity Search
        ↓
Top-k Similar Support Tickets
     
```


###### Section 1 — Import Libraries

In [0]:
# -----------------------------
# Install once if needed
# -----------------------------
%pip install databricks-vectorsearch
dbutils.library.restartPython()



In [0]:

# -----------------------------
# Import libraries
# -----------------------------
from databricks.vector_search.client import VectorSearchClient
from databricks.sdk import WorkspaceClient


###### Section 2 — Configuration

In [0]:
# -----------------------------
# Define names
# -----------------------------
SOURCE_TABLE = "dbw_agentic_ai_dev.support_ticket_ai.gold_support_ticket_embeddings"

VECTOR_SEARCH_ENDPOINT_NAME = "support-ticket-vector-search-endpoint"

INDEX_NAME = (
    "dbw_agentic_ai_dev.support_ticket_ai."
    "gold_support_ticket_embeddings_index"
)

EMBEDDING_MODEL_NAME = "databricks-gte-large-en"

###### Section 3 — Enable Change Data Feed

In [0]:
# -----------------------------
# Enable Change Data Feed
# Required for Delta Sync Index
# -----------------------------
spark.sql(f"""
ALTER TABLE {SOURCE_TABLE}
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)
""")

###### Section 4 — Validate Source Table

In [0]:
# -----------------------------
# Validate source table
# -----------------------------
source_df = spark.table(SOURCE_TABLE)

total_count = source_df.count()
distinct_count = source_df.select("ticket_id").distinct().count()

print("Total rows:", total_count)
print("Distinct ticket_id:", distinct_count)

if total_count != distinct_count:
    raise ValueError("ticket_id must be unique because it is used as primary key.")


# -----------------------------
# Get embedding dimension dynamically
# -----------------------------
embedding_dimension = len(
    source_df
    .select("embedding")
    .first()["embedding"]
)

print("Embedding dimension:", embedding_dimension)




###### Section 5 — Create Vector Search Endpoint

In [0]:
# -----------------------------
# Create Vector Search client
# -----------------------------
vsc = VectorSearchClient()


# -----------------------------
# Create Vector Search Endpoint
# If it already exists, continue
# -----------------------------
try:
    vsc.create_endpoint(
        name=VECTOR_SEARCH_ENDPOINT_NAME,
        endpoint_type="STANDARD"
    )
    print("Vector Search endpoint created.")
except Exception as e:
    print("Vector Search endpoint may already exist.")
    print(e)


###### Section 6 — Create Delta Sync Index

In [0]:
# -----------------------------
# Create Delta Sync Index
# If it already exists, continue
# -----------------------------
try:
    index = vsc.create_delta_sync_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        source_table_name=SOURCE_TABLE,
        index_name=INDEX_NAME,
        pipeline_type="TRIGGERED",
        primary_key="ticket_id",
        embedding_dimension=embedding_dimension,
        embedding_vector_column="embedding",
        columns_to_sync=[
            "ticket_id",
            "cleaned_ticket_text",
            "category"
        ]
    )
    print("Delta Sync Index created.")
except Exception as e:
    print("Delta Sync Index may already exist.")
    print(e)

    index = vsc.get_index(
        endpoint_name=VECTOR_SEARCH_ENDPOINT_NAME,
        index_name=INDEX_NAME
    )

# -----------------------------
# Synchronize the Delta Sync Index with the latest
# changes from the Delta table.
# -----------------------------
index.sync()
print("Index sync triggered.")


###### Section 7 — Create Helper Function

In [0]:
# -----------------------------
# Create Workspace client
# Used to call embedding model endpoint
# -----------------------------
w = WorkspaceClient()


# -----------------------------
# Helper function for semantic search
# -----------------------------
def semantic_search(question, num_results=3):
    """
    Convert a user question into an embedding using the same embedding model,
    then search the Vector Search index for similar support tickets.
    """

    # Generate embedding for user question
    response = w.serving_endpoints.query(
        name=EMBEDDING_MODEL_NAME,
        input=[question]
    )

    question_embedding = [
        float(x) for x in response.data[0].embedding
    ]

    # Query Vector Search index
    results = index.similarity_search(
        query_vector=question_embedding,
        columns=[
            "ticket_id",
            "cleaned_ticket_text",
            "category"
        ],
        num_results=num_results
    )

    return results

###### Section 8 — Test

In [0]:
# -----------------------------
# Test semantic search
# -----------------------------
test_questions = [
    "router is not working",
    "refund not received",
    "cancel my subscription",
    "cannot access my customer portal"
]

for question in test_questions:
    print("\nQuestion:", question)
    results = semantic_search(question, num_results=3)
    print(results)

###### Notebook Summary

- Created a Vector Search Endpoint.

- Created a Delta Sync Index from the embedding table.

- Generated embeddings for user questions.

- Retrieved semantically similar support tickets using Vector Search.


###### Key Learnings

- Why Vector Search requires embeddings instead of raw text.

- Difference between Vector Search Endpoint and Delta Sync Index.

- Why Change Data Feed is required.

- Importance of using the same embedding model for indexing and querying.

- Similarity search compares meanings, not keywords.

- Delta Sync Index automatically keeps the Vector Index synchronized with Delta tables.

- Vector Search returns the nearest vectors using cosine similarity.

- Semantic Search retrieves documents with similar meaning even when the words differ.


###### Notebook Conclusion

- In this notebook, we built a semantic retrieval layer by creating a Databricks Vector Search Endpoint and Delta Sync Index over the support ticket embeddings.

- This enables semantic search by retrieving the most relevant support tickets based on meaning rather than exact keyword matching.

- This will be used in the next notebook to retrieve relevant support ticket context for a Retrieval-Augmented Generation (RAG) application.


###### Next Notebook

24_nlp_RAG

- The purpose of this notebook is to combine Vector Search with a Large Language Model (LLM) to answer user questions using retrieved support ticket information.